<a href="https://colab.research.google.com/github/ShaneHurley/BasketballElo/blob/main/BasketballEloSim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import warnings
import itertools
from tqdm.auto import tqdm
from scipy.sparse import lil_matrix, csr_matrix
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, log_loss

warnings.filterwarnings('ignore')

# -------------------------------
# 1. BASE SYSTEM
# -------------------------------
BASE_ELO = 1500
ELO_SCALING_FACTOR = 1000

# -------------------------------
# 2. HYPERPARAMETERS
# -------------------------------
K_OFF=0.3
K_DEF=0.7
CONFIDENCE = 400
ASSIST_SPLIT = 0.65  # Shooter gets 65%, Assister gets 35%

# -------------------------------
# 3. PROJECTION BIASES & CAPS
# -------------------------------
HOME_PPP_BOOST = 0.024
O_MULT_CAP = 2.5
D_MULT_CAP = 2.2
USAGE_FLOOR = 0.15

# -------------------------------
# 4. HUSTLE WEIGHTS & BASELINE
# -------------------------------
WEIGHT_OREB = 0.0    # Set dynamically in Cell 7
WEIGHT_TOV = 0.0
WEIGHT_FOUL_DRAWN = 0.0
WEIGHT_BLK = 0.0
LEAGUE_PPP = 1.10
BACK_TO_BACK_PENALTY = 0.95

In [2]:
class PlayerRatingTracker:
    def __init__(self, confidence=CONFIDENCE):
        self.players = {}

    def get_player(self, pid):
        if pid not in self.players:
            self.players[pid] = {'O_Elo': BASE_ELO, 'D_Elo': BASE_ELO,
                                 'Possessions': 0, 'Cumulative_Residual': 0.0}
        return self.players[pid]

    def get_lineup_rating(self, player_ids):
        """Returns the pure baseline Elo of the 5-man unit (No Synergy here)."""
        active_ids = sorted([pid for pid in player_ids if pid and str(pid) != 'nan'])
        n_active = len(active_ids)
        base_w = 1.0 / n_active if n_active > 0 else 0.2

        o_sum = sum(self.get_player(pid)['O_Elo'] for pid in active_ids)
        d_sum = sum(self.get_player(pid)['D_Elo'] for pid in active_ids)
        avg_exp = np.mean([self.get_player(pid)['Possessions'] for pid in active_ids]) if active_ids else 0

        return {
            'O_Elo': o_sum * base_w,
            'D_Elo': d_sum * base_w,
            'Avg_Exp': avg_exp,
            'Base_Weight': base_w
        }, [base_w] * n_active

    def update_ratings(self, player_ids, o_shift, d_shift, poss,
                       off_usage, def_usage, base_w, team_residual):
        active_ids = [pid for pid in player_ids if pid and str(pid) != 'nan']
        if not active_ids: return

        total_o_use = sum(off_usage.values())
        total_d_use = sum(def_usage.values())

        o_shares, d_shares = [], []
        for pid in active_ids:
            raw_o = (off_usage.get(pid, 0) / total_o_use) if total_o_use > 0 else base_w
            o_shares.append(max(raw_o, base_w * USAGE_FLOOR))

            # Defense is 80% team credit, 20% box-score individual credit
            raw_d = (def_usage.get(pid, 0) / total_d_use) if total_d_use > 0 else base_w
            d_shares.append((0.80 * base_w) + (0.20 * raw_d))

        sum_o, sum_d = sum(o_shares), sum(d_shares)
        o_shares = [s / sum_o for s in o_shares]
        d_shares = [s / sum_d for s in d_shares]

        for i, pid in enumerate(active_ids):
            p = self.get_player(pid)
            p['O_Elo'] += (o_shift * 0.90 * (o_shares[i] / base_w))
            p['D_Elo'] += (d_shift * 0.90 * (d_shares[i] / base_w))
            p['Possessions'] += poss
            p['Cumulative_Residual'] += (team_residual * base_w)

    def apply_offseason_regression(self, player_ages, player_three_rate):
        for pid, data in self.players.items():
            age = player_ages.get(pid, 27)
            three_rate = player_three_rate.get(pid, 0.25)

            # Age-based momentum/decline
            if age <= 23:
                base = 0.85
            elif age >= 33:
                base = 0.60
            else:
                base = 0.75

            # Archetype boost (shooters age better)
            arch_boost = 0.05 if three_rate > 0.40 else (-0.05 if three_rate < 0.15 else 0.0)
            carryover = np.clip(base + arch_boost, 0.55, 0.90)

            data['O_Elo'] = (data['O_Elo'] * carryover) + (BASE_ELO * (1 - carryover))
            data['D_Elo'] = (data['D_Elo'] * carryover) + (BASE_ELO * (1 - carryover))
            data['Possessions'] = 0
            data['Cumulative_Residual'] = 0.0

In [3]:
def process_stint(tracker, ids_A, ids_B, poss, pts_A, pts_B,
                  usage_A, usage_B, def_usage_A, def_usage_B,
                  tovs_A, blks_A, tovs_B, blks_B,
                  oreb_A, oreb_B, fouls_drawn_A, fouls_drawn_B,
                  period, start_score_A, start_score_B, end_score_A, end_score_B,
                  b2b_penalty_A=1.0, b2b_penalty_B=1.0,
                  k_off=0.5, k_def=0.5, league_avg_ppp=1.10, synergy_dicts=None):
    if poss <= 0: return 0.0, 0.0

    rating_A, _ = tracker.get_lineup_rating(ids_A)
    rating_B, _ = tracker.get_lineup_rating(ids_B)

    # Base Expected PPP
    xPPP_A = league_avg_ppp + HOME_PPP_BOOST + ((rating_A['O_Elo'] - rating_B['D_Elo']) / ELO_SCALING_FACTOR)
    xPPP_B = league_avg_ppp - HOME_PPP_BOOST + ((rating_B['O_Elo'] - rating_A['D_Elo']) / ELO_SCALING_FACTOR)

    # Apply Optional HAPM Synergy directly to xPPP (Do NOT pollute Elo ratings)
    if synergy_dicts and len(synergy_dicts) == 3:
        dyad_syn, triad_syn, quintet_syn = synergy_dicts
        syn_A, syn_B = 0.0, 0.0

        active_A = [p for p in ids_A if p and p != 'nan']
        active_B = [p for p in ids_B if p and p != 'nan']

        for pair in itertools.combinations(active_A, 2): syn_A += dyad_syn.get(f"{pair[0]}-{pair[1]}", 0.0)
        for trip in itertools.combinations(active_A, 3): syn_A += triad_syn.get(f"{trip[0]}-{trip[1]}-{trip[2]}", 0.0)
        if len(active_A) == 5: syn_A += quintet_syn.get("-".join(sorted(active_A)), 0.0)

        for pair in itertools.combinations(active_B, 2): syn_B += dyad_syn.get(f"{pair[0]}-{pair[1]}", 0.0)
        for trip in itertools.combinations(active_B, 3): syn_B += triad_syn.get(f"{trip[0]}-{trip[1]}-{trip[2]}", 0.0)
        if len(active_B) == 5: syn_B += quintet_syn.get("-".join(sorted(active_B)), 0.0)

        xPPP_A += syn_A
        xPPP_B += syn_B

    xPPP_A = np.clip(xPPP_A, 0, 3)
    xPPP_B = np.clip(xPPP_B, 0, 3)

    # Restored Hustle Weights (eff_pts)
    eff_pts_A = pts_A + (oreb_A * WEIGHT_OREB) + (fouls_drawn_A * WEIGHT_FOUL_DRAWN) - (tovs_A * WEIGHT_TOV) - (blks_B * WEIGHT_BLK)
    eff_pts_B = pts_B + (oreb_B * WEIGHT_OREB) + (fouls_drawn_B * WEIGHT_FOUL_DRAWN) - (tovs_B * WEIGHT_TOV) - (blks_A * WEIGHT_BLK)

    res_A = eff_pts_A - (xPPP_A * poss)
    res_B = eff_pts_B - (xPPP_B * poss)

    avg_exp = min(rating_A['Avg_Exp'], rating_B['Avg_Exp'])
    k_mult = np.clip(avg_exp / 500.0, 0.2, 1.0)

    # Leverage Scaling
    lead = abs(end_score_A - end_score_B)
    if period > 4:
        leverage = 2.0
    else:
        urgency = period / 4.0
        leverage = np.clip(0.5 + 1.5 * np.exp(-0.05 * lead) * urgency, 0.3, 2.0)

    # ----- STINT-LEVEL CLUTCH WIN BONUS -----
    clutch_A, clutch_B = 1.0, 1.0
    if period >= 4 and lead <= 7:
        if res_A > 0: clutch_A = 1.4 if lead <= 3 else 1.2
        if res_B > 0: clutch_B = 1.4 if lead <= 3 else 1.2

    # Chaos Dampener
    chaos_damp = 0.7 if poss > 25 else (0.9 if poss > 18 else 1.0)

    # Apply all dynamic multipliers
    dynamic_k_off_A = k_off * k_mult * leverage * chaos_damp * clutch_A * b2b_penalty_A
    dynamic_k_def_A = k_def * k_mult * leverage * chaos_damp * clutch_A * b2b_penalty_A

    dynamic_k_off_B = k_off * k_mult * leverage * chaos_damp * clutch_B * b2b_penalty_B
    dynamic_k_def_B = k_def * k_mult * leverage * chaos_damp * clutch_B * b2b_penalty_B

    o_shift_A = dynamic_k_off_A * res_A
    d_shift_A = dynamic_k_def_A * (-res_B)
    o_shift_B = dynamic_k_off_B * res_B
    d_shift_B = dynamic_k_def_B * (-res_A)

    tracker.update_ratings(ids_A, o_shift_A, d_shift_A, poss, usage_A, def_usage_A, rating_A['Base_Weight'], res_A)
    tracker.update_ratings(ids_B, o_shift_B, d_shift_B, poss, usage_B, def_usage_B, rating_B['Base_Weight'], res_B)

    # Return residuals so game-level bonus can accumulate them
    return res_A, res_B

In [4]:
def format_players(team, row):
    players = [row[f"{team}_PLAYER_ID_{i+1}"] for i in range(5)]
    players = [int(p) if pd.notna(p) else None for p in players]
    players = [str(p) for p in players if p is not None]
    players.sort()
    return "-".join(players)

def preprocess_pbp(pbp_df):
    is_home_event = pbp_df['HOMEDESCRIPTION'].notna() & pbp_df['VISITORDESCRIPTION'].isna()
    is_away_event = pbp_df['VISITORDESCRIPTION'].notna() & pbp_df['HOMEDESCRIPTION'].isna()

    pbp_df['is_fg_make'] = (pbp_df['EVENTMSGTYPE'] == 1).astype(int)
    pbp_df['is_fg_miss'] = (pbp_df['EVENTMSGTYPE'] == 2).astype(int)
    pbp_df['is_tov'] = (pbp_df['EVENTMSGTYPE'] == 5).astype(int)
    pbp_df['is_foul'] = (pbp_df['EVENTMSGTYPE'] == 6).astype(int)

    combo_desc = pbp_df['HOMEDESCRIPTION'].fillna('') + " " + pbp_df['VISITORDESCRIPTION'].fillna('')
    is_final_ft = ((pbp_df['EVENTMSGTYPE'] == 3) &
                   combo_desc.str.contains(r'(1 of 1|2 of 2|3 of 3)', regex=True, na=False))
    is_tech = ((pbp_df['EVENTMSGTYPE'] == 3) & (pbp_df.get('EVENTMSGACTIONTYPE', 0) == 16))
    is_and1 = ((pbp_df['EVENTMSGTYPE'] == 3) &
               (pbp_df['EVENTMSGTYPE'].shift(1) == 1) &
               (pbp_df['PLAYER1_ID'] == pbp_df['PLAYER1_ID'].shift(1)))
    pbp_df['is_true_ft_trip'] = (is_final_ft & ~is_and1 & ~is_tech).astype(int)

    pbp_df['home_pts_added'] = 0
    pbp_df['away_pts_added'] = 0
    home_make_mask = (pbp_df['is_fg_make'] == 1) & is_home_event
    away_make_mask = (pbp_df['is_fg_make'] == 1) & is_away_event
    pbp_df.loc[home_make_mask, 'home_pts_added'] = pbp_df.loc[home_make_mask, 'HOMEDESCRIPTION'].apply(
        lambda x: 3 if '3PT' in str(x) else 2)
    pbp_df.loc[away_make_mask, 'away_pts_added'] = pbp_df.loc[away_make_mask, 'VISITORDESCRIPTION'].apply(
        lambda x: 3 if '3PT' in str(x) else 2)

    home_ft_mask = (pbp_df['EVENTMSGTYPE'] == 3) & is_home_event & (~combo_desc.str.contains('MISS', na=False))
    away_ft_mask = (pbp_df['EVENTMSGTYPE'] == 3) & is_away_event & (~combo_desc.str.contains('MISS', na=False))
    pbp_df.loc[home_ft_mask, 'home_pts_added'] = 1
    pbp_df.loc[away_ft_mask, 'away_pts_added'] = 1

    pbp_df['shot_team_state'] = np.nan
    pbp_df.loc[(pbp_df['is_fg_miss'] == 1) & is_home_event, 'shot_team_state'] = 'home'
    pbp_df.loc[(pbp_df['is_fg_miss'] == 1) & is_away_event, 'shot_team_state'] = 'away'
    reset_mask = pbp_df['EVENTMSGTYPE'].isin([1, 5, 8]) | (pbp_df['PERIOD'] != pbp_df['PERIOD'].shift(1))
    pbp_df.loc[reset_mask, 'shot_team_state'] = 'RESET'
    pbp_df['last_shot_team'] = pbp_df['shot_team_state'].ffill()
    pbp_df.loc[pbp_df['last_shot_team'] == 'RESET', 'last_shot_team'] = np.nan

    pbp_df['home_oreb'] = ((pbp_df['EVENTMSGTYPE'] == 4) & is_home_event &
                           (pbp_df['last_shot_team'] == 'home')).astype(int)
    pbp_df['away_oreb'] = ((pbp_df['EVENTMSGTYPE'] == 4) & is_away_event &
                           (pbp_df['last_shot_team'] == 'away')).astype(int)

    pbp_df['home_tovs_forced'] = ((pbp_df['is_tov'] == 1) & is_away_event).astype(int)
    pbp_df['away_tovs_forced'] = ((pbp_df['is_tov'] == 1) & is_home_event).astype(int)

    if 'PLAYER3_ID' in pbp_df.columns:
        pbp_df['home_blks_forced'] = ((pbp_df['is_fg_miss'] == 1) & is_away_event &
                                      pbp_df['PLAYER3_ID'].notna()).astype(int)
        pbp_df['away_blks_forced'] = ((pbp_df['is_fg_miss'] == 1) & is_home_event &
                                      pbp_df['PLAYER3_ID'].notna()).astype(int)
    else:
        pbp_df['home_blks_forced'] = 0
        pbp_df['away_blks_forced'] = 0

    pbp_df['home_fouls_drawn'] = ((pbp_df['is_foul'] == 1) & is_away_event).astype(int)
    pbp_df['away_fouls_drawn'] = ((pbp_df['is_foul'] == 1) & is_home_event).astype(int)

    pbp_df['home_poss'] = ((pbp_df['is_fg_make'] + pbp_df['is_fg_miss'] + pbp_df['is_tov']) *
                           is_home_event.astype(int) - pbp_df['home_oreb'] +
                           0.44 * pbp_df['is_true_ft_trip'] * is_home_event.astype(int))
    pbp_df['away_poss'] = ((pbp_df['is_fg_make'] + pbp_df['is_fg_miss'] + pbp_df['is_tov']) *
                           is_away_event.astype(int) - pbp_df['away_oreb'] +
                           0.44 * pbp_df['is_true_ft_trip'] * is_away_event.astype(int))
    pbp_df['total_possessions'] = (pbp_df['home_poss'] + pbp_df['away_poss']) / 2

    return pbp_df

def prepare_game_outcomes(stints_df):
    games = {}
    for _, s in stints_df.iterrows():
        gid = s['GAME_ID']
        if gid not in games:
            games[gid] = {
                'home_players': str(s['HOME_players']).split('-'),
                'away_players': str(s['AWAY_players']).split('-'),
                'home_score': 0, 'away_score': 0
            }
        games[gid]['home_score'] += s['home_pts']
        games[gid]['away_score'] += s['away_pts']
    records = []
    for gid, data in games.items():
        records.append({
            'GAME_ID': gid,
            'home_players': data['home_players'], 'away_players': data['away_players'],
            'home_score': data['home_score'], 'away_score': data['away_score'],
            'home_win': 1 if data['home_score'] > data['away_score'] else 0
        })
    return pd.DataFrame(records)

In [5]:
def build_stints_and_usage(pbp_df):
    pbp_df['stint_id'] = (
        (pbp_df['HOME_players'] != pbp_df['HOME_players'].shift()) |
        (pbp_df['AWAY_players'] != pbp_df['AWAY_players'].shift()) |
        (pbp_df['PERIOD'] != pbp_df['PERIOD'].shift())
    ).cumsum()

    # NOTE: ALL hustle stats are correctly grouped here to prevent KeyErrors
    stints_df = pbp_df.groupby('stint_id').agg(
        GAME_ID=('GAME_ID', 'first'), PERIOD=('PERIOD', 'first'),
        HOME_players=('HOME_players', 'first'), AWAY_players=('AWAY_players', 'first'),
        HOME_SCORE_START=('HOME_SCORE', 'first'), AWAY_SCORE_START=('AWAY_SCORE', 'first'),
        home_pts=('home_pts_added', 'sum'), away_pts=('away_pts_added', 'sum'),
        home_oreb=('home_oreb', 'sum'), away_oreb=('away_oreb', 'sum'),
        home_tovs=('home_tovs_forced', 'sum'), away_tovs=('away_tovs_forced', 'sum'),
        home_blks_against=('home_blks_forced', 'sum'), away_blks_against=('away_blks_forced', 'sum'),
        home_fouls_drawn=('home_fouls_drawn', 'sum'), away_fouls_drawn=('away_fouls_drawn', 'sum'),
        possessions=('total_possessions', 'sum')
    ).reset_index()

    stints_df['HOME_SCORE_END'] = stints_df['HOME_SCORE_START'] + stints_df['home_pts']
    stints_df['AWAY_SCORE_END'] = stints_df['AWAY_SCORE_START'] + stints_df['away_pts']

    home_u, away_u, home_d, away_d = {}, {}, {}, {}
    for _, row in stints_df.iterrows():
        sid = row['stint_id']
        home_u[sid] = {p: 0.0 for p in [p for p in str(row['HOME_players']).split('-') if p and p != 'nan']}
        away_u[sid] = {p: 0.0 for p in [p for p in str(row['AWAY_players']).split('-') if p and p != 'nan']}
        home_d[sid] = {p: 0.0 for p in home_u[sid].keys()}
        away_d[sid] = {p: 0.0 for p in away_u[sid].keys()}

    records = pbp_df.to_dict('records')
    for i, row in enumerate(records):
        stint = row['stint_id']
        p1 = str(int(row['PLAYER1_ID'])) if pd.notna(row.get('PLAYER1_ID')) else None
        p2 = str(int(row['PLAYER2_ID'])) if pd.notna(row.get('PLAYER2_ID')) else None
        p3 = str(int(row['PLAYER3_ID'])) if pd.notna(row.get('PLAYER3_ID')) else None

        is_make = row.get('is_fg_make', 0) == 1
        is_miss = row.get('is_fg_miss', 0) == 1
        is_tov = row.get('is_tov', 0) == 1
        is_ft = row.get('is_true_ft_trip', 0) == 1
        evt = row.get('EVENTMSGTYPE')

        next_is_oreb = False
        if is_miss and i + 1 < len(records) and records[i + 1].get('EVENTMSGTYPE') == 4 and records[i + 1].get('PLAYER1_TEAM_ID') == row.get('PLAYER1_TEAM_ID'):
            next_is_oreb = True

        hu, au, hd, ad = home_u.get(stint, {}), away_u.get(stint, {}), home_d.get(stint, {}), away_d.get(stint, {})

        if is_make:
            # 65/35 ASSIST SPLIT REWARDING PLAYMAKERS
            if p2 in hu and p1 in hu:
                hu[p1] += ASSIST_SPLIT
                hu[p2] += (1.0 - ASSIST_SPLIT)
            elif p1 in hu:
                hu[p1] += 1.0

            if p2 in au and p1 in au:
                au[p1] += ASSIST_SPLIT
                au[p2] += (1.0 - ASSIST_SPLIT)
            elif p1 in au:
                au[p1] += 1.0
        elif is_miss and not next_is_oreb:
            if p1 in hu: hu[p1] += 1.0
            if p1 in au: au[p1] += 1.0
        elif is_tov or is_ft:
            if p1 in hu: hu[p1] += 1.0
            if p1 in au: au[p1] += 1.0

        if evt == 4:
            if p1 in hd and row.get('home_oreb', 0) == 0: hd[p1] += 1.0
            if p1 in ad and row.get('away_oreb', 0) == 0: ad[p1] += 1.0
        elif evt == 5:
            if p2 in hd: hd[p2] += 1.5
            if p2 in ad: ad[p2] += 1.5
        elif evt == 2:
            if p3 in hd: hd[p3] += 1.5
            if p3 in ad: ad[p3] += 1.5

    stints_df['home_usage'] = stints_df['stint_id'].map(home_u)
    stints_df['away_usage'] = stints_df['stint_id'].map(away_u)
    stints_df['home_def_usage'] = stints_df['stint_id'].map(home_d)
    stints_df['away_def_usage'] = stints_df['stint_id'].map(away_d)
    return stints_df[stints_df['possessions'] > 0].reset_index(drop=True)

In [6]:
from google.colab import drive
drive.mount('/drive', force_remount=True)

names_df = pd.read_csv(r"/drive/MyDrive/basketballData/BasketballELO/PlayerList08.csv", low_memory=False)
names_dict = names_df.set_index('PERSON_ID').to_dict()['DISPLAY_FIRST_LAST']

start_years = [2013, 2014, 2015, 2016, 2017] # Customize to your full dataset range
all_stints_dict = {}
game_outcomes_dict = {}

for start_year in start_years:
    end_year = start_year + 1
    print(f"\nProcessing {start_year}-{end_year}...")
    file_path = f"/drive/MyDrive/basketballData/BasketballELO/events_{start_year}-{end_year}_pbp.csv"
    pbp_df = pd.read_csv(file_path, low_memory=False)

    pbp_df['HOME_players'] = pbp_df.apply(lambda r: format_players('HOME', r), axis=1)
    pbp_df['AWAY_players'] = pbp_df.apply(lambda r: format_players('AWAY', r), axis=1)

    pbp_df = preprocess_pbp(pbp_df)
    stints_df = build_stints_and_usage(pbp_df)

    all_stints_dict[start_year] = stints_df
    game_outcomes_dict[start_year] = prepare_game_outcomes(stints_df)

master_stints = pd.concat(all_stints_dict.values(), ignore_index=True)
print("\nAll seasons preprocessed and loaded into dictionaries.")

def build_archetype_dicts(master_stints):
    # In production, merge explicit Birthdates from PlayerList08.csv here.
    player_ages = {}
    player_three_rate = {}
    return player_ages, player_three_rate

Mounted at /drive

Processing 2013-2014...

Processing 2014-2015...

Processing 2015-2016...

Processing 2016-2017...

Processing 2017-2018...

All seasons preprocessed and loaded into dictionaries.


In [7]:
total_pts = master_stints['home_pts'].sum() + master_stints['away_pts'].sum()
total_poss = master_stints['possessions'].sum() * 2
LEAGUE_PPP = total_pts / total_poss if total_poss > 0 else 1.10

WEIGHT_OREB = LEAGUE_PPP * 1.10
WEIGHT_TOV = LEAGUE_PPP
WEIGHT_FOUL_DRAWN = 1.55 - LEAGUE_PPP
WEIGHT_BLK = LEAGUE_PPP * 0.55

print(f"\n=== ERA-ADJUSTED HUSTLE WEIGHTS ===")
print(f"League PPP: {LEAGUE_PPP:.3f}")
print(f"OREB: +{WEIGHT_OREB:.3f} pts")
print(f"TOV:  -{WEIGHT_TOV:.3f} pts")
print(f"FOUL: +{WEIGHT_FOUL_DRAWN:.3f} pts")
print(f"BLK:  +{WEIGHT_BLK:.3f} pts")


=== ERA-ADJUSTED HUSTLE WEIGHTS ===
League PPP: 1.307
OREB: +1.437 pts
TOV:  -1.307 pts
FOUL: +0.243 pts
BLK:  +0.719 pts


In [8]:
def predict_game(tracker, home_ids, away_ids, neutral=False, synergy_dicts=None):
    rating_home, _ = tracker.get_lineup_rating(home_ids)
    rating_away, _ = tracker.get_lineup_rating(away_ids)

    xPPP_A = LEAGUE_PPP + HOME_PPP_BOOST + ((rating_home['O_Elo'] - rating_away['D_Elo']) / ELO_SCALING_FACTOR)
    xPPP_B = LEAGUE_PPP - HOME_PPP_BOOST + ((rating_away['O_Elo'] - rating_home['D_Elo']) / ELO_SCALING_FACTOR)

    if synergy_dicts and len(synergy_dicts) == 3:
        dyad_syn, triad_syn, quintet_syn = synergy_dicts
        syn_A, syn_B = 0.0, 0.0

        active_A = [p for p in home_ids if p and p != 'nan']
        for pair in itertools.combinations(active_A, 2): syn_A += dyad_syn.get(f"{pair[0]}-{pair[1]}", 0.0)
        for trip in itertools.combinations(active_A, 3): syn_A += triad_syn.get(f"{trip[0]}-{trip[1]}-{trip[2]}", 0.0)
        if len(active_A) == 5: syn_A += quintet_syn.get("-".join(sorted(active_A)), 0.0)

        active_B = [p for p in away_ids if p and p != 'nan']
        for pair in itertools.combinations(active_B, 2): syn_B += dyad_syn.get(f"{pair[0]}-{pair[1]}", 0.0)
        for trip in itertools.combinations(active_B, 3): syn_B += triad_syn.get(f"{trip[0]}-{trip[1]}-{trip[2]}", 0.0)
        if len(active_B) == 5: syn_B += quintet_syn.get("-".join(sorted(active_B)), 0.0)

        xPPP_A += syn_A
        xPPP_B += syn_B

    net_ppp = xPPP_A - xPPP_B
    if neutral: net_ppp -= 2 * HOME_PPP_BOOST

    margin_per_100 = net_ppp * 100
    win_prob = 1.0 / (1.0 + 10.0 ** (-margin_per_100 / 15.0))

    return {'home_win_probability': win_prob, 'expected_margin_per_100': margin_per_100}

def test_model_accuracy(tracker, test_year, all_stints_dict, game_outcomes_dict, synergy_dicts=None):
    test_stints = all_stints_dict[test_year]
    test_games = game_outcomes_dict[test_year]

    game_expected_margins, game_actual_margins = {}, {}

    for _, s in test_stints.iterrows():
        ids_A = str(s['HOME_players']).split('-')
        ids_B = str(s['AWAY_players']).split('-')

        pred = predict_game(tracker, ids_A, ids_B, neutral=False, synergy_dicts=synergy_dicts)
        expected_margin_ppp = pred['expected_margin_per_100'] / 100.0

        gid = s['GAME_ID']
        if gid not in game_expected_margins:
            game_expected_margins[gid], game_actual_margins[gid] = 0, 0

        game_expected_margins[gid] += expected_margin_ppp * s['possessions']
        game_actual_margins[gid] += (s['home_pts'] - s['away_pts'])

    game_probs, game_labels, abs_errors = [], [], []

    for _, g in test_games.iterrows():
        gid = g['GAME_ID']
        total_expected_margin = game_expected_margins.get(gid, 0)
        actual_margin = game_actual_margins.get(gid, 0)

        abs_errors.append(abs(total_expected_margin - actual_margin))
        win_prob = 1.0 / (1.0 + 10.0 ** (-total_expected_margin / 15.0))
        game_probs.append(win_prob)
        game_labels.append(g['home_win'])

    game_mae = np.mean(abs_errors)
    logloss = log_loss(game_labels, game_probs)
    return game_mae, logloss

def calibrate_k_multi_year(train_years, test_year, all_stints_dict, game_outcomes_dict, k_grid=[0.1, 0.3, 0.5, 0.7]):
    """
    Grid searches hyperparameters by training sequentially across multiple years
    before testing on the final isolated year.
    """
    print(f"Calibrating K factors (Train Years: {train_years} | Test Year: {test_year}) ...")
    best_logloss = float('inf')
    best_params = None

    for k_off in k_grid:
        for k_def in k_grid:
            tracker = PlayerRatingTracker(confidence=CONFIDENCE)

            # --- MULTI-YEAR TRAINING LOOP ---
            for year in train_years:
                stints = all_stints_dict[year]

                for _, s in stints.iterrows():
                    # We intentionally ignore residuals during calibration loops
                    _, _ = process_stint(tracker,
                        ids_A=str(s['HOME_players']).split('-'), ids_B=str(s['AWAY_players']).split('-'),
                        poss=s['possessions'], pts_A=s['home_pts'], pts_B=s['away_pts'],
                        usage_A=s['home_usage'], usage_B=s['away_usage'],
                        def_usage_A=s['home_def_usage'], def_usage_B=s['away_def_usage'],
                        tovs_A=s['home_tovs'], blks_A=s['home_blks_against'],
                        tovs_B=s['away_tovs'], blks_B=s['away_blks_against'],
                        oreb_A=s['home_oreb'], oreb_B=s['away_oreb'],
                        fouls_drawn_A=s['home_fouls_drawn'], fouls_drawn_B=s['away_fouls_drawn'],
                        period=s['PERIOD'], start_score_A=s['HOME_SCORE_START'], start_score_B=s['AWAY_SCORE_START'],
                        end_score_A=s['HOME_SCORE_END'], end_score_B=s['AWAY_SCORE_END'],
                        k_off=k_off, k_def=k_def, league_avg_ppp=LEAGUE_PPP, synergy_dicts=None)

                # Apply offseason regression between training years
                # (Passing empty dicts {} here uses the baseline 0.75 age-default to save memory during heavy calibration loops)
                if year != train_years[-1]:
                    tracker.apply_offseason_regression({}, {})

            # --- ISOLATED TESTING ---
            mae, ll = test_model_accuracy(tracker, test_year, all_stints_dict, game_outcomes_dict, synergy_dicts=None)
            print(f"K_OFF={k_off}, K_DEF={k_def} -> Game MAE: {mae:.2f} | LogLoss: {ll:.4f}")

            if ll < best_logloss:
                best_logloss = ll
                best_params = (k_off, k_def)

    return best_params, best_logloss

# ---------------------------------------------------------
# EXECUTION EXAMPLE:
# Uncomment the code below if you want to run the calibration.
# ---------------------------------------------------------

train_seasons = [2013, 2014, 2015, 2016, 2017]
test_season = 2017
optimal_k, best_ll = calibrate_k_multi_year(train_seasons, test_season, all_stints_dict, game_outcomes_dict, k_grid=[0.1, 0.3, 0.5, 0.7])
print(f"\nOPTIMAL: K_OFF={optimal_k[0]}, K_DEF={optimal_k[1]}, LogLoss={best_ll:.4f}")

Calibrating K factors (Train Years: [2013, 2014, 2015, 2016, 2017] | Test Year: 2017) ...
K_OFF=0.1, K_DEF=0.1 -> Game MAE: 10.02 | LogLoss: 0.6331
K_OFF=0.1, K_DEF=0.3 -> Game MAE: 9.88 | LogLoss: 0.6196
K_OFF=0.1, K_DEF=0.5 -> Game MAE: 9.95 | LogLoss: 0.6229
K_OFF=0.1, K_DEF=0.7 -> Game MAE: 10.05 | LogLoss: 0.6297
K_OFF=0.3, K_DEF=0.1 -> Game MAE: 10.05 | LogLoss: 0.6275
K_OFF=0.3, K_DEF=0.3 -> Game MAE: 9.90 | LogLoss: 0.6122
K_OFF=0.3, K_DEF=0.5 -> Game MAE: 9.95 | LogLoss: 0.6127
K_OFF=0.3, K_DEF=0.7 -> Game MAE: 10.04 | LogLoss: 0.6172
K_OFF=0.5, K_DEF=0.1 -> Game MAE: 10.17 | LogLoss: 0.6309
K_OFF=0.5, K_DEF=0.3 -> Game MAE: 10.05 | LogLoss: 0.6163
K_OFF=0.5, K_DEF=0.5 -> Game MAE: 10.09 | LogLoss: 0.6162
K_OFF=0.5, K_DEF=0.7 -> Game MAE: 10.18 | LogLoss: 0.6205
K_OFF=0.7, K_DEF=0.1 -> Game MAE: 10.30 | LogLoss: 0.6364
K_OFF=0.7, K_DEF=0.3 -> Game MAE: 10.22 | LogLoss: 0.6228
K_OFF=0.7, K_DEF=0.5 -> Game MAE: 10.25 | LogLoss: 0.6227
K_OFF=0.7, K_DEF=0.7 -> Game MAE: 10.34 | Lo

In [9]:
def build_synergy_pyramid(train_stints, alpha=100.0):
    # High Alpha prevents data noise from dominating the Elo baseline
    print("Building Sparse HAPM Design Matrix...")

    all_players, all_dyads, all_triads, all_quintets = set(), set(), set(), set()

    for _, row in train_stints.iterrows():
        home = sorted([p for p in row['HOME_players'].split('-') if p and p != 'nan'])
        away = sorted([p for p in row['AWAY_players'].split('-') if p and p != 'nan'])
        for lineup in (home, away):
            if len(lineup) != 5: continue
            all_players.update(lineup)
            for pair in itertools.combinations(lineup, 2): all_dyads.add(f"{pair[0]}-{pair[1]}")
            for trip in itertools.combinations(lineup, 3): all_triads.add(f"{trip[0]}-{trip[1]}-{trip[2]}")
            all_quintets.add("-".join(lineup))

    p_idx = {p: i for i, p in enumerate(sorted(all_players))}
    d_idx = {d: i for i, d in enumerate(sorted(all_dyads))}
    t_idx = {t: i for i, t in enumerate(sorted(all_triads))}
    q_idx = {q: i for i, q in enumerate(sorted(all_quintets))}

    n_p, n_d, n_t, n_q = len(all_players), len(all_dyads), len(all_triads), len(all_quintets)

    X_sparse = lil_matrix((len(train_stints), n_p + n_d + n_t + n_q), dtype=np.int8)
    y, w = np.zeros(len(train_stints)), np.zeros(len(train_stints))

    for i, (_, row) in enumerate(train_stints.iterrows()):
        home = sorted([p for p in row['HOME_players'].split('-') if p and p != 'nan'])
        away = sorted([p for p in row['AWAY_players'].split('-') if p and p != 'nan'])

        for p in home: X_sparse[i, p_idx[p]] = 1
        for p in away: X_sparse[i, p_idx[p]] = -1
        for pair in itertools.combinations(home, 2): X_sparse[i, n_p + d_idx[f"{pair[0]}-{pair[1]}"]] = 1
        for pair in itertools.combinations(away, 2): X_sparse[i, n_p + d_idx[f"{pair[0]}-{pair[1]}"]] = -1
        for trip in itertools.combinations(home, 3): X_sparse[i, n_p + n_d + t_idx[f"{trip[0]}-{trip[1]}-{trip[2]}"]] = 1
        for trip in itertools.combinations(away, 3): X_sparse[i, n_p + n_d + t_idx[f"{trip[0]}-{trip[1]}-{trip[2]}"]] = -1
        if len(home) == 5: X_sparse[i, n_p + n_d + n_t + q_idx["-".join(home)]] = 1
        if len(away) == 5: X_sparse[i, n_p + n_d + n_t + q_idx["-".join(away)]] = -1

        poss = row['possessions']
        if poss > 0:
            y[i] = (row['home_pts'] - row['away_pts']) / poss
            w[i] = np.sqrt(poss)

    X_csr = X_sparse.tocsr()
    print(f"Fitting Ridge Regression with {X_csr.shape[1]} features...")
    model = Ridge(alpha=alpha, fit_intercept=True).fit(X_csr, y, sample_weight=w)

    # We deliberately discard the Individual RAPM elements here. Elo handles the base.
    return (
        {d: model.coef_[n_p + i] for i, d in enumerate(sorted(all_dyads))},
        {t: model.coef_[n_p + n_d + i] for i, t in enumerate(sorted(all_triads))},
        {q: model.coef_[n_p + n_d + n_t + i] for i, q in enumerate(sorted(all_quintets))}
    )

In [10]:
tracker = PlayerRatingTracker(confidence=CONFIDENCE)
synergy_cache = {}

player_ages, player_three_rate = build_archetype_dicts(master_stints)

for year in start_years:
    # Build Rolling Synergy from PAST years only
    past_years = [y for y in start_years if y < year]
    if len(past_years) >= 2:
        train_df = pd.concat([all_stints_dict[y] for y in past_years[-2:]], ignore_index=True)
        synergy_cache[year] = build_synergy_pyramid(train_df, alpha=100.0)
    else:
        synergy_cache[year] = ({}, {}, {})

for year in start_years:
    stints = all_stints_dict[year]
    print(f"\nSimulating {year}-{year+1}...")

    synergy_dicts = synergy_cache.get(year)
    game_residuals = {}

    stints_records = stints.to_dict('records')
    for s in tqdm(stints_records, desc=f"Season {year}"):
        gid = s['GAME_ID']
        if gid not in game_residuals:
            game_residuals[gid] = {'home_res': 0.0, 'away_res': 0.0, 'home_players': set(), 'away_players': set()}

        h_players = [p for p in str(s['HOME_players']).split('-') if p and p != 'nan']
        a_players = [p for p in str(s['AWAY_players']).split('-') if p and p != 'nan']

        game_residuals[gid]['home_players'].update(h_players)
        game_residuals[gid]['away_players'].update(a_players)

        res_A, res_B = process_stint(tracker,
            ids_A=h_players, ids_B=a_players,
            poss=s['possessions'], pts_A=s['home_pts'], pts_B=s['away_pts'],
            usage_A=s['home_usage'], usage_B=s['away_usage'],
            def_usage_A=s['home_def_usage'], def_usage_B=s['away_def_usage'],
            tovs_A=s['home_tovs'], blks_A=s['home_blks_against'],
            tovs_B=s['away_tovs'], blks_B=s['away_blks_against'],
            oreb_A=s['home_oreb'], oreb_B=s['away_oreb'],
            fouls_drawn_A=s['home_fouls_drawn'], fouls_drawn_B=s['away_fouls_drawn'],
            period=s['PERIOD'], start_score_A=s['HOME_SCORE_START'], start_score_B=s['AWAY_SCORE_START'],
            end_score_A=s['HOME_SCORE_END'], end_score_B=s['AWAY_SCORE_END'],
            b2b_penalty_A=1.0, b2b_penalty_B=1.0,
            k_off=K_OFF, k_def=K_DEF, league_avg_ppp=LEAGUE_PPP, synergy_dicts=synergy_dicts)

        game_residuals[gid]['home_res'] += res_A
        game_residuals[gid]['away_res'] += res_B

    # ----- GAME-LEVEL CLUTCH WIN BONUS -----
    game_outcomes = game_outcomes_dict[year]
    for _, g in game_outcomes.iterrows():
        gid = g['GAME_ID']
        home_win = g['home_win']
        final_margin = abs(g['home_score'] - g['away_score'])

        if final_margin <= 5 and gid in game_residuals:
            if home_win:
                bonus = 0.2 * max(0, game_residuals[gid]['home_res'])
                players = list(game_residuals[gid]['home_players'])
            else:
                bonus = 0.2 * max(0, game_residuals[gid]['away_res'])
                players = list(game_residuals[gid]['away_players'])

            if players and bonus > 0:
                per_player = bonus / len(players)
                for pid in players:
                    p = tracker.get_player(pid)
                    p['O_Elo'] += per_player / 2
                    p['D_Elo'] += per_player / 2

    # Offseason Reset
    if year != start_years[-1]:
        tracker.apply_offseason_regression(player_ages, player_three_rate)

final_ratings = tracker
print("\nAll seasons simulated securely.")
if len(start_years) > 1:
    mae, ll = test_model_accuracy(final_ratings, start_years[-1], all_stints_dict, game_outcomes_dict, synergy_dicts=synergy_cache.get(start_years[-1]))

Building Sparse HAPM Design Matrix...
Fitting Ridge Regression with 61393 features...
Building Sparse HAPM Design Matrix...
Fitting Ridge Regression with 61165 features...
Building Sparse HAPM Design Matrix...
Fitting Ridge Regression with 58502 features...

Simulating 2013-2014...


Season 2013:   0%|          | 0/34493 [00:00<?, ?it/s]


Simulating 2014-2015...


Season 2014:   0%|          | 0/35724 [00:00<?, ?it/s]


Simulating 2015-2016...


...


Simulating 2016-2017...


Season 2016:   0%|          | 0/35575 [00:00<?, ?it/s]


Simulating 2017-2018...


Season 2017:   0%|          | 0/35587 [00:00<?, ?it/s]


All seasons simulated securely.


In [11]:
results = []
for pid, data in final_ratings.players.items():
    if data['Possessions'] < 800:
        continue
    total_elo = data['O_Elo'] + data['D_Elo']
    impact_100 = (total_elo - 3000) / (ELO_SCALING_FACTOR / 100)
    try:
        name = names_dict.get(int(pid), str(pid))
    except:
        name = str(pid)
    results.append({
        'Player': name,
        'Total_Elo': round(total_elo, 1),
        'Impact_100': round(impact_100, 2),
        'O_Elo': round(data['O_Elo'], 1),
        'D_Elo': round(data['D_Elo'], 1),
        'Possessions': int(data['Possessions'])
    })

rankings_df = pd.DataFrame(results).sort_values('Total_Elo', ascending=False)
print("\n=== TOP 20 PLAYERS BY TOTAL ELO ===")
print(rankings_df.head(20).to_string(index=False))


=== TOP 20 PLAYERS BY TOTAL ELO ===
               Player  Total_Elo  Impact_100  O_Elo  D_Elo  Possessions
          Rudy Gobert     3854.0       85.40 1710.1 2143.9         2955
       DeAndre Jordan     3805.6       80.56 1728.9 2076.7         4044
         Nikola Jokic     3747.0       74.70 1651.8 2095.2         4039
         James Harden     3701.3       70.13 2050.4 1650.9         4203
         LeBron James     3642.3       64.23 1864.8 1777.5         5163
        Dwight Howard     3632.0       63.20 1585.7 2046.3         4183
        Anthony Davis     3606.6       60.66 1498.8 2107.8         4759
         Jimmy Butler     3573.6       57.36 1746.7 1826.9         3508
    Russell Westbrook     3562.6       56.26 1742.6 1819.9         4886
    LaMarcus Aldridge     3549.6       54.96 1545.0 2004.5         4057
          Aron Baynes     3546.3       54.63 1465.9 2080.4         2479
     Hassan Whiteside     3545.4       54.54 1576.5 1968.9         2239
         Kevin Durant     3

In [12]:
def display_hapm_rankings(synergy_dicts, names_dict, top_n=10):
    if not synergy_dicts or len(synergy_dicts) != 3:
        print("No synergy dictionaries available for extraction.")
        return

    dyad_syn, triad_syn, quintet_syn = synergy_dicts

    def get_names(id_str):
        return " + ".join([names_dict.get(int(pid), str(pid)) for pid in id_str.split('-') if pid.isdigit()])

    dyad_list = [{'2-Man Group': get_names(k), 'Synergy (per 100)': v * 100} for k, v in dyad_syn.items()]
    dyad_df = pd.DataFrame(dyad_list).sort_values('Synergy (per 100)', ascending=False)
    print(f"\n=== TOP {top_n} 2-MAN GROUPS ===")
    print(dyad_df.head(top_n).to_string(index=False))

    triad_list = [{'3-Man Group': get_names(k), 'Synergy (per 100)': v * 100} for k, v in triad_syn.items()]
    triad_df = pd.DataFrame(triad_list).sort_values('Synergy (per 100)', ascending=False)
    print(f"\n=== TOP {top_n} 3-MAN GROUPS ===")
    print(triad_df.head(top_n).to_string(index=False))

    quintet_list = [{'5-Man Group': get_names(k), 'Synergy (per 100)': v * 100} for k, v in quintet_syn.items()]
    quintet_df = pd.DataFrame(quintet_list).sort_values('Synergy (per 100)', ascending=False)
    print(f"\n=== TOP {top_n} 5-MAN GROUPS ===")
    print(quintet_df.head(top_n).to_string(index=False))

latest_year = start_years[-1] if 'start_years' in locals() and start_years else None
if latest_year and latest_year in synergy_cache:
    display_hapm_rankings(synergy_cache[latest_year], names_dict, top_n=10)


=== TOP 10 2-MAN GROUPS ===
                     2-Man Group  Synergy (per 100)
Bobby Portis + Cristiano Felicio          16.983563
     Tyus Jones + Andrew Wiggins          16.861966
  Eric Bledsoe + Leandro Barbosa          15.068183
     Kyle Singler + Jerami Grant          14.449895
  Jose Calderon + Taurean Prince          14.242408
 Derrick Jones Jr. + T.J. Warren          13.421480
    Kevin Love + DeAndre Liggins          13.129389
  Paul Pierce + Luc Mbah a Moute          13.061960
    Paul Zipser + Nikola Mirotic          12.791664
   Richaun Holmes + Phil Pressey          12.640938

=== TOP 10 3-MAN GROUPS ===
                                      3-Man Group  Synergy (per 100)
  Malcolm Delaney + Kent Bazemore + Dwight Howard          20.667122
        Evan Turner + Tyler Zeller + Marcus Smart          20.226938
       Vince Carter + Jeff Green + JaMychal Green          19.630905
    LaMarcus Aldridge + Danny Green + Patty Mills          18.374027
         Vince Carter + M

In [17]:
import pandas as pd
import numpy as np
import itertools
import xgboost as xgb
from sklearn.metrics import log_loss, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

# -------------------------------------------------------------------------
# 1. DYNAMIC PACE TRACKING ARCHITECTURE
# -------------------------------------------------------------------------
class PaceTracker:
    def __init__(self, window=15):
        self.team_paces = {}
        self.league_paces = []
        self.window = window

    def update_pace(self, team_hash, game_possessions):
        if team_hash not in self.team_paces:
            self.team_paces[team_hash] = []
        self.team_paces[team_hash].append(game_possessions)
        self.league_paces.append(game_possessions)
        if len(self.team_paces[team_hash]) > self.window:
            self.team_paces[team_hash].pop(0)

    def get_team_pace(self, team_hash, default_pace):
        if team_hash in self.team_paces and len(self.team_paces[team_hash]) > 0:
            return sum(self.team_paces[team_hash]) / len(self.team_paces[team_hash])
        return default_pace

    def predict_matchup_pace(self, home_hash, away_hash, fallback_pace=100.0):
        league_avg = sum(self.league_paces[-self.window*15:]) / len(self.league_paces[-self.window*15:]) if self.league_paces else fallback_pace
        home_pace = self.get_team_pace(home_hash, league_avg)
        away_pace = self.get_team_pace(away_hash, league_avg)
        return (home_pace * away_pace) / league_avg if league_avg > 0 else fallback_pace

# -------------------------------------------------------------------------
# 2. FULL ROSTER PREDICTION MODEL
# -------------------------------------------------------------------------
def predict_full_game_roster(tracker, home_roster_shares, away_roster_shares,
                             expected_possessions=100.0, synergy_dicts=None, home_court=True, prob_divisor=15.0):

    home_total = sum(home_roster_shares.values())
    away_total = sum(away_roster_shares.values())

    if home_total == 0 or away_total == 0:
        return {
            'home_win_prob': 0.5, 'expected_spread': 0.0, 'margin_per_100': 0.0,
            'home_off_rating': LEAGUE_PPP*100, 'away_off_rating': LEAGUE_PPP*100
        }

    def get_safe_team_averages(shares, total_volume):
        off_sum, def_sum = 0.0, 0.0
        for pid, volume in shares.items():
            if volume > 0:
                weight = volume / total_volume
                p = tracker.players.get(str(pid), {'O_Elo': BASE_ELO, 'D_Elo': BASE_ELO})
                off_sum += p['O_Elo'] * weight
                def_sum += p['D_Elo'] * weight
        return off_sum, def_sum

    home_off, home_def = get_safe_team_averages(home_roster_shares, home_total)
    away_off, away_def = get_safe_team_averages(away_roster_shares, away_total)

    syn_home, syn_away = 0.0, 0.0
    if synergy_dicts and len(synergy_dicts) == 3:
        dyad_syn, triad_syn, _ = synergy_dicts

        active_home = [p for p, vol in home_roster_shares.items() if vol > 0]
        for p1, p2 in itertools.combinations(active_home, 2):
            pair_key = f"{p1}-{p2}" if int(p1) < int(p2) else f"{p2}-{p1}"
            if pair_key in dyad_syn:
                p1_floor = min(1.0, 5.0 * home_roster_shares[p1] / home_total)
                p2_floor = min(1.0, 5.0 * home_roster_shares[p2] / home_total)
                syn_home += dyad_syn[pair_key] * (p1_floor * p2_floor)

        active_away = [p for p, vol in away_roster_shares.items() if vol > 0]
        for p1, p2 in itertools.combinations(active_away, 2):
            pair_key = f"{p1}-{p2}" if int(p1) < int(p2) else f"{p2}-{p1}"
            if pair_key in dyad_syn:
                p1_floor = min(1.0, 5.0 * away_roster_shares[p1] / away_total)
                p2_floor = min(1.0, 5.0 * away_roster_shares[p2] / away_total)
                syn_away += dyad_syn[pair_key] * (p1_floor * p2_floor)

    home_off_rating = (LEAGUE_PPP * 100 + (home_off - away_def) / (ELO_SCALING_FACTOR / 100) + (HOME_PPP_BOOST * 100 if home_court else 0) + syn_home * 100)
    away_off_rating = (LEAGUE_PPP * 100 + (away_off - home_def) / (ELO_SCALING_FACTOR / 100) - (HOME_PPP_BOOST * 100 if home_court else 0) + syn_away * 100)

    home_off_rating, away_off_rating = np.clip(home_off_rating, 70, 140), np.clip(away_off_rating, 70, 140)

    margin_per_100 = home_off_rating - away_off_rating
    expected_spread = (margin_per_100 / 100.0) * expected_possessions
    home_win_prob = 1.0 / (1.0 + 10.0 ** (-margin_per_100 / prob_divisor))

    # This MUST explicitly return the dictionary to prevent the NoneType error!
    return {
        'home_win_prob': home_win_prob,
        'expected_spread': expected_spread,
        'margin_per_100': margin_per_100,
        'home_off_rating': home_off_rating,
        'away_off_rating': away_off_rating
    }

# -------------------------------------------------------------------------
# 3. BACKTEST VALIDATOR (Generates XGBoost Matrix)
# -------------------------------------------------------------------------
def evaluate_game_predictions(tracker, stints_df, game_outcomes_df, synergy_dicts=None, prob_divisor=15.0):
    game_player_poss = {}

    for _, s in stints_df.iterrows():
        gid = s['GAME_ID']
        poss = s['possessions']
        if poss <= 0: continue

        if gid not in game_player_poss:
            game_player_poss[gid] = {'home': {}, 'away': {}, 'total_poss': 0}

        for p in [x for x in str(s['HOME_players']).split('-') if x and x != 'nan']:
            game_player_poss[gid]['home'][p] = game_player_poss[gid]['home'].get(p, 0) + poss
        for p in [x for x in str(s['AWAY_players']).split('-') if x and x != 'nan']:
            game_player_poss[gid]['away'][p] = game_player_poss[gid]['away'].get(p, 0) + poss
        game_player_poss[gid]['total_poss'] += poss

    results = []
    for _, g in game_outcomes_df.iterrows():
        gid = g['GAME_ID']
        if gid not in game_player_poss: continue

        pred = predict_full_game_roster(
            tracker, game_player_poss[gid]['home'], game_player_poss[gid]['away'],
            expected_possessions=game_player_poss[gid]['total_poss'],
            synergy_dicts=synergy_dicts,
            prob_divisor=prob_divisor
        )

        actual_spread = g['home_score'] - g['away_score']
        actual_home_won = g['home_win']
        pred_home_won = 1 if pred['expected_spread'] > 0 else 0

        results.append({
            'GAME_ID': gid,
            'Expected_Spread': round(pred['expected_spread'], 2),
            'Actual_Spread': actual_spread,
            'Spread_Error': abs(pred['expected_spread'] - actual_spread),
            'Home_Win_Prob': pred['home_win_prob'],
            'Predicted_Win': pred_home_won,
            'Actual_Win': actual_home_won,
            'Correct_Winner': 1 if pred_home_won == actual_home_won else 0,
            'home_off_rating': pred['home_off_rating'],
            'away_off_rating': pred['away_off_rating'],
            'home_def_rating': pred['away_off_rating'],
            'away_def_rating': pred['home_off_rating'],
            'margin_per_100': pred['margin_per_100'],
            'expected_possessions': game_player_poss[gid]['total_poss']
        })

    res_df = pd.DataFrame(results)
    return res_df

# -------------------------------------------------------------------------
# 4. EXECUTE EVALUATION AND TRAIN XGBOOST
# -------------------------------------------------------------------------
# Run the evaluation to generate the feature matrix
optimal_div = 15.0 # Use the optimal divisor if you ran grid search, otherwise default to 15.0
backtest_metrics_df = evaluate_game_predictions(
    clean_tracker,
    test_stints_data,
    test_outcomes_data,
    synergy_dicts=test_synergy_map,
    prob_divisor=optimal_div
)

print(f"\n=== BASE ELO BACKTEST RESULTS ===")
print(f"Games Evaluated : {len(backtest_metrics_df)}")
print(f"Spread MAE      : {backtest_metrics_df['Spread_Error'].mean():.2f} pts")
print(f"Win/Loss Acc    : {backtest_metrics_df['Correct_Winner'].mean() * 100:.2f}%")

print("\n=== XGBoost Meta-Model Training ===")
features = [
    'home_off_rating', 'away_off_rating',
    'home_def_rating', 'away_def_rating',
    'margin_per_100', 'expected_possessions',
    'Expected_Spread'
]
target = 'Actual_Spread'

X = backtest_metrics_df[features]
y = backtest_metrics_df[target]

tscv = TimeSeriesSplit(n_splits=5)
fold_maes, fold_elo_maes = [], []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[test_idx]

    xgb_model = xgb.XGBRegressor(
        n_estimators=150, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, objective='reg:squarederror', random_state=42
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    val_preds = xgb_model.predict(X_val)
    mae_xgb = mean_absolute_error(y_val, val_preds)
    mae_elo = mean_absolute_error(y_val, X_val['Expected_Spread'])

    fold_maes.append(mae_xgb)
    fold_elo_maes.append(mae_elo)
    print(f"Fold {fold+1} | XGBoost MAE: {mae_xgb:.2f} | Base Elo MAE: {mae_elo:.2f}")

print("-" * 45)
print(f"Average XGBoost MAE : {np.mean(fold_maes):.2f} pts")
print(f"Average Base Elo MAE: {np.mean(fold_elo_maes):.2f} pts")
print(f"Net Improvement     : {np.mean(fold_elo_maes) - np.mean(fold_maes):.2f} pts")

final_xgb_model = xgb.XGBRegressor(
    n_estimators=150, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8, objective='reg:squarederror', random_state=42
)
final_xgb_model.fit(X, y)
print("\nFinal Meta-Model Trained and Ready for Live Inference.")


=== BASE ELO BACKTEST RESULTS ===
Games Evaluated : 1230
Spread MAE      : 11.54 pts
Win/Loss Acc    : 62.52%

=== XGBoost Meta-Model Training ===
Fold 1 | XGBoost MAE: 11.22 | Base Elo MAE: 11.39
Fold 2 | XGBoost MAE: 10.76 | Base Elo MAE: 10.84
Fold 3 | XGBoost MAE: 11.28 | Base Elo MAE: 11.98
Fold 4 | XGBoost MAE: 11.53 | Base Elo MAE: 13.05
Fold 5 | XGBoost MAE: 9.85 | Base Elo MAE: 10.80
---------------------------------------------
Average XGBoost MAE : 10.93 pts
Average Base Elo MAE: 11.61 pts
Net Improvement     : 0.68 pts

Final Meta-Model Trained and Ready for Live Inference.


In [18]:
# 1. Setup True Out-Of-Sample Environment
test_season_year = start_years[-1]
train_season_years = [y for y in start_years if y < test_season_year]

print(f"Training on: {train_season_years} | Testing purely on: {test_season_year}")

clean_tracker = PlayerRatingTracker(confidence=CONFIDENCE)
pace_tracker = PaceTracker()
clean_player_ages, clean_player_three_rate = build_archetype_dicts(master_stints)
synergy_cache = {}

# 2. Train the Model on Past Years
for year in train_season_years:
    stints = all_stints_dict[year]

    # Build Synergy strictly from past data prior to the current training year
    past_years = [y for y in train_season_years if y < year]
    if len(past_years) >= 2:
        train_df = pd.concat([all_stints_dict[y] for y in past_years[-2:]], ignore_index=True)
        synergy_cache[year] = build_synergy_pyramid(train_df, alpha=100.0)
    else:
        synergy_cache[year] = ({}, {}, {})

    print(f"Simulating {year}-{year+1}...")
    for _, s in tqdm(stints.iterrows(), total=len(stints), desc=f"Season {year}"):
        _, _ = process_stint(
            clean_tracker,
            ids_A=str(s['HOME_players']).split('-'), ids_B=str(s['AWAY_players']).split('-'),
            poss=s['possessions'], pts_A=s['home_pts'], pts_B=s['away_pts'],
            usage_A=s['home_usage'], usage_B=s['away_usage'],
            def_usage_A=s['home_def_usage'], def_usage_B=s['away_def_usage'],
            tovs_A=s['home_tovs'], blks_A=s['home_blks_against'],
            tovs_B=s['away_tovs'], blks_B=s['away_blks_against'],
            oreb_A=s['home_oreb'], oreb_B=s['away_oreb'],
            fouls_drawn_A=s['home_fouls_drawn'], fouls_drawn_B=s['away_fouls_drawn'],
            period=s['PERIOD'], start_score_A=s['HOME_SCORE_START'], start_score_B=s['AWAY_SCORE_START'],
            end_score_A=s['HOME_SCORE_END'], end_score_B=s['AWAY_SCORE_END'],
            k_off=K_OFF, k_def=K_DEF, league_avg_ppp=LEAGUE_PPP, synergy_dicts=synergy_cache.get(year)
        )

    if year != train_season_years[-1]:
        clean_tracker.apply_offseason_regression(clean_player_ages, clean_player_three_rate)

# 3. Test the Model on the Unseen Target Year
print(f"\n--- INITIATING UNSEEN PRE-GAME BACKTEST ({test_season_year}) ---")
test_stints_data = all_stints_dict[test_season_year]
test_outcomes_data = game_outcomes_dict[test_season_year]

# Build synergy map strictly from the end of the training period for the test year
if len(train_season_years) >= 2:
    test_train_df = pd.concat([all_stints_dict[y] for y in train_season_years[-2:]], ignore_index=True)
    test_synergy_map = build_synergy_pyramid(test_train_df, alpha=100.0)
else:
    test_synergy_map = ({}, {}, {})

# Evaluate
backtest_metrics_df = evaluate_game_predictions(
    clean_tracker,
    test_stints_data,
    test_outcomes_data,
    synergy_dicts=test_synergy_map
)

print("\nSample Game Forecast Logs:")
print(backtest_metrics_df[['GAME_ID', 'Expected_Spread', 'Actual_Spread', 'Spread_Error', 'Home_Win_Prob']].head(15).to_string(index=False))

Training on: [2013, 2014, 2015, 2016] | Testing purely on: 2017
Simulating 2013-2014...


Season 2013:   0%|          | 0/34493 [00:00<?, ?it/s]

Simulating 2014-2015...


Season 2014:   0%|          | 0/35724 [00:00<?, ?it/s]

Building Sparse HAPM Design Matrix...
Fitting Ridge Regression with 61393 features...
Simulating 2015-2016...


Season 2015:   0%|          | 0/35991 [00:00<?, ?it/s]

Building Sparse HAPM Design Matrix...
Fitting Ridge Regression with 61165 features...
Simulating 2016-2017...


Season 2016:   0%|          | 0/35575 [00:00<?, ?it/s]


--- INITIATING UNSEEN PRE-GAME BACKTEST (2017) ---
Building Sparse HAPM Design Matrix...
Fitting Ridge Regression with 58502 features...

Sample Game Forecast Logs:
 GAME_ID  Expected_Spread  Actual_Spread  Spread_Error  Home_Win_Prob
21700874            11.64             23     11.356793       0.890063
21700014            16.94             15      1.938368       0.960380
21701230            -1.15             13     14.145448       0.445245
21700482             8.47              5      3.472209       0.823492
21700877             7.92             17      9.078292       0.823984
21700355            18.73              3     15.726595       0.958798
21700608            -1.09              9     10.087022       0.451633
21700609            -3.49              2      5.490826       0.330192
21700610            16.48              7      9.482527       0.951509
21700611            -5.27             18     23.266000       0.257042
21700612             7.36            -15     22.362955       0.8

In [19]:
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

print("=== XGBoost Meta-Model Training ===")

# 1. Define Features and Targets using the Backtester output
features = [
    'home_off_rating', 'away_off_rating',
    'home_def_rating', 'away_def_rating',
    'margin_per_100', 'expected_possessions',
    'Expected_Spread' # The original Elo baseline prediction
]
target = 'Actual_Spread'

X = backtest_metrics_df[features]
y = backtest_metrics_df[target]

# 2. Time Series Cross-Validation
tscv = TimeSeriesSplit(n_splits=5)
fold_maes = []
fold_elo_maes = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[test_idx]

    # Initialize XGBoost Regressor
    xgb_model = xgb.XGBRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42
    )

    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    # Predict and evaluate
    val_preds = xgb_model.predict(X_val)
    mae_xgb = mean_absolute_error(y_val, val_preds)
    mae_elo = mean_absolute_error(y_val, X_val['Expected_Spread'])

    fold_maes.append(mae_xgb)
    fold_elo_maes.append(mae_elo)
    print(f"Fold {fold+1} | XGBoost MAE: {mae_xgb:.2f} | Base Elo MAE: {mae_elo:.2f}")

print("-" * 45)
print(f"Average XGBoost MAE : {np.mean(fold_maes):.2f} pts")
print(f"Average Base Elo MAE: {np.mean(fold_elo_maes):.2f} pts")
print(f"Net Improvement     : {np.mean(fold_elo_maes) - np.mean(fold_maes):.2f} pts")

# -------------------------------------------------------------------------
# 3. ENSEMBLE BLENDING & FINAL INFERENCE MODEL
# -------------------------------------------------------------------------
# Train the master model on all available data for live predictions
final_xgb_model = xgb.XGBRegressor(
    n_estimators=150, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8, objective='reg:squarederror', random_state=42
)
final_xgb_model.fit(X, y)

def predict_live_game_ensemble(tracker, pace_tracker, home_id, away_id, home_shares, away_shares, prob_divisor=15.0):
    """
    Generates a live blended prediction using Elo and XGBoost.
    """
    expected_poss = pace_tracker.predict_

=== XGBoost Meta-Model Training ===
Fold 1 | XGBoost MAE: 11.22 | Base Elo MAE: 11.39
Fold 2 | XGBoost MAE: 10.76 | Base Elo MAE: 10.84
Fold 3 | XGBoost MAE: 11.28 | Base Elo MAE: 11.98
Fold 4 | XGBoost MAE: 11.53 | Base Elo MAE: 13.05
Fold 5 | XGBoost MAE: 9.85 | Base Elo MAE: 10.80
---------------------------------------------
Average XGBoost MAE : 10.93 pts
Average Base Elo MAE: 11.61 pts
Net Improvement     : 0.68 pts
